# Libera - Kernels Inspection Example

This notebook demonstrates how to inspect the dynamic kernels provided by Libera using libera_utils` and `curryer` libraries. We will load the kernels, extract ephemeris and attitude data, and visualize them.

This example uses test data available in this repository.

In [ ]:
import logging
from pathlib import Path
import importlib
import os

import numpy as np
import xarray as xr
import matplotlib.pyplot \
    as plt
import matplotlib.dates as mdates

from curryer import utils, spicetime, meta
from curryer import spicierpy as sp
from libera_utils.libera_spice.kernel_manager import KernelManager

DYNAMIC_KERNEL_PATH = Path(os.getcwd()).parent / "tests/test_data/dynamic_kernels"

xr.set_options(display_width=120, display_max_rows=30)
np.set_printoptions(linewidth=120)
utils.enable_logging(log_level=logging.DEBUG, extra_loggers=[__name__])
figsize = (12, 6)

In [ ]:
# Using KernelManager to load kernels and furnish them to SpiceyPy
# This will load static kernels, NAIF kernels, and Libera dynamic kernels.
# This allows SpiceyPy or SpicierPy calls (from curryer) to access them and inspect them.

km = KernelManager()
km.load_static_kernels()
km.load_naif_kernels()
km.load_libera_dynamic_kernels(DYNAMIC_KERNEL_PATH)

## Inspect JPSS

In [ ]:
from libera_utils.libera_spice.spice_utils import ls_all_kernel_coverage

ls_all_kernel_coverage()

In [ ]:
utc_range = ('2028-01-02T00:00:00', '2028-01-02T00:30:00')
et_range = spicetime.adapt(utc_range, 'iso', 'et')
et_times = np.arange(*et_range, 10)[1:-1]
dt64_times = spicetime.adapt(et_times, 'et', 'dt64')

In [ ]:
positions = []
for sample_et in et_times:
    sample_arr, _ = sp.spkezr(
        sp.obj.Body('JPSS4_SC').id,
        sample_et,
        ref='ITRF93',
        abcorr='NONE',
        obs=sp.obj.Body('EARTH').id,
    )
    positions.append(sample_arr)
positions = np.vstack(positions)
velocities = positions[:, 3:]
positions = positions[:, :3]

In [ ]:
fig, ax = plt.subplots(figsize=figsize)

for ith, field in enumerate(['X', 'Y', 'Z']):
    ax.scatter(dt64_times, positions[:, ith], s=1, label=field)

ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))  # '%Y-%m-%d'
ax.legend() ; ax.grid()
ax.set_title('Libera - Ephemeris (ECEF)');

In [ ]:
fig, ax = plt.subplots(figsize=figsize)

for ith, field in enumerate(['VX', 'VY', 'VZ']):
    ax.scatter(dt64_times, velocities[:, ith], s=1, label=field)

ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))  # '%Y-%m-%d'
ax.legend() ; ax.grid()
ax.set_title('Libera - Velocity (ECEF)');

In [ ]:
attitude = []
for sample_et in et_times:
    sample_arr = sp.pxform('JPSS4_SC_COORD', 'ITRF93', sample_et)
    sample_arr = sp.m2q(sample_arr)
    attitude.append(sample_arr)
attitude = np.vstack(attitude)

In [ ]:
fig, ax = plt.subplots(figsize=figsize)

for ith, field in enumerate(['S', 'I', 'J', 'K']):
    ax.scatter(dt64_times, attitude[:, ith], s=1, label=field)

ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))  # '%Y-%m-%d'
ax.legend() ; ax.grid()
ax.set_title('Libera - Attitude (ECEF)');

## Inspect AzEl

In [ ]:
utc_range2 = ('2028-01-02T00:13:47', '2028-01-02T00:29:00')
et_range2 = spicetime.adapt(utc_range2, 'iso', 'et')
et_times2 = np.arange(*et_range2, 1)[1:-1]
dt64_times2 = spicetime.adapt(et_times2, 'et', 'dt64')

In [ ]:
azrot = []
for sample_et in et_times2:
    sample_arr = sp.pxform('LIBERA_AZ_COORD', 'LIBERA_BASE_COORD', sample_et)
    sample_arr = sp.m2eul(sample_arr, 3, 2, 1)
    sample_arr = np.rad2deg(sample_arr[0])
    azrot.append(sample_arr)
azrot = np.vstack(azrot)

In [ ]:
fig, ax = plt.subplots(figsize=figsize)

ax.scatter(dt64_times2, azrot, s=1, label='Az')

ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))  # '%Y-%m-%d'
ax.legend() ; ax.grid()
ax.set_title('Libera - Az Rotation (deg)');

In [ ]:
elrot = []
for sample_et in et_times2:
    sample_arr = sp.pxform('LIBERA_EL_COORD', 'LIBERA_AZ_COORD', sample_et)
    sample_arr = sp.m2eul(sample_arr, 3, 2, 1)
    sample_arr = np.rad2deg(sample_arr[2])
    elrot.append(sample_arr)
elrot = np.vstack(elrot)

In [ ]:
fig, ax = plt.subplots(figsize=figsize)

ax.scatter(dt64_times2, elrot, s=1, label='El')

ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))  # '%Y-%m-%d'
ax.legend() ; ax.grid()
ax.set_title('Libera - El Rotation (deg)');